Gold layer customer_360

In [0]:
%sql
 CREATE OR REPLACE TABLE customer_360.gold.customer_360 AS
 WITH
 base AS (
   SELECT
     customer_id,
     first_name,
     last_name,
     full_name,
     email,
     city,
     credit_score,
     credit_category,
     credit_rating,
     created_at,
     customer_tenure_days,
     customer_since_year
   FROM customer_360.silver.silver_customers
 ),

 account_metrics AS (
   SELECT
     customer_id,
     COUNT(*)                                                 AS total_accounts,
     ROUND(SUM(balance_usd), 2)                              AS total_balance,
     ROUND(SUM(CASE WHEN account_type = 'Savings'
                    THEN balance_usd ELSE 0 END), 2)         AS savings_balance,
     ROUND(SUM(CASE WHEN account_type = 'Checking'
                    THEN balance_usd ELSE 0 END), 2)         AS checking_balance,
     ROUND(SUM(CASE WHEN account_type = 'Business'
                    THEN balance_usd ELSE 0 END), 2)         AS business_balance,
     ROUND(AVG(account_age_days), 0)                         AS avg_account_age_days,
     
     CASE MAX(CASE balance_tier
               WHEN 'Platinum' THEN 4
               WHEN 'Gold'     THEN 3
               WHEN 'Silver'   THEN 2
               ELSE                 1
             END)
       WHEN 4 THEN 'Platinum'
       WHEN 3 THEN 'Gold'
       WHEN 2 THEN 'Silver'
       ELSE        'Standard'
     END                                                      AS top_balance_tier
   FROM customer_360.silver.silver_accounts
   GROUP BY customer_id
 ),

 loan_metrics AS (
   SELECT
     customer_id,
     COUNT(*)                                                 AS total_loans,
     ROUND(SUM(loan_amount), 2)                              AS total_loan_amount,
     ROUND(AVG(interest_rate), 2)                            AS avg_interest_rate,
     MAX(CASE WHEN is_high_interest THEN 1 ELSE 0 END)       AS has_high_interest_loan,
     ROUND(SUM(monthly_interest_estimate), 2)                AS total_monthly_interest
   FROM customer_360.silver.silver_loans
   GROUP BY customer_id
 ),

 card_metrics AS (
   SELECT
     sa.customer_id,
     COUNT(*)                                                 AS total_cards,
     SUM(CASE WHEN sc.card_status = 'Active' THEN 1 ELSE 0 END) AS active_cards,
     MAX(CASE WHEN sc.card_type = 'Credit'
              AND sc.card_status = 'Active' THEN 1 ELSE 0 END) AS has_active_credit_card,
     MAX(CASE WHEN sc.card_type = 'Debit'
              AND sc.card_status = 'Active' THEN 1 ELSE 0 END) AS has_active_debit_card
   FROM customer_360.silver.silver_cards sc
   JOIN customer_360.silver.silver_accounts sa
     ON sc.account_id = sa.account_id
   GROUP BY sa.customer_id
 ),

 txn_metrics AS (
   SELECT
     customer_id,
     COUNT(*)                                                 AS total_transactions,
     SUM(CASE WHEN is_successful THEN 1 ELSE 0 END)          AS successful_transactions,
     ROUND(SUM(spending_amount), 2)                          AS total_spending,
     COUNT(DISTINCT year_month)                              AS active_months,
     ROUND(
       SUM(spending_amount) / NULLIF(COUNT(DISTINCT year_month), 0),
     2)                                                      AS avg_monthly_spending,
     MAX(transaction_date)                                   AS last_transaction_date,
     DATEDIFF(current_date(), MAX(transaction_date))         AS days_since_last_transaction,
     SUM(CASE WHEN is_digital_channel THEN 1 ELSE 0 END)     AS digital_transactions,
     SUM(CASE WHEN is_card_transaction THEN 1 ELSE 0 END)    AS card_transactions
   FROM customer_360.silver.silver_transactions
   GROUP BY customer_id
 ),

 digital_metrics AS (
   SELECT
     customer_id,
     COUNT(*)                                                 AS total_sessions,
     SUM(CASE WHEN is_login            THEN 1 ELSE 0 END)    AS total_logins,
     SUM(CASE WHEN is_mobile           THEN 1 ELSE 0 END)    AS mobile_sessions,
     SUM(CASE WHEN NOT is_mobile       THEN 1 ELSE 0 END)    AS desktop_sessions,
     ROUND(
       SUM(CASE WHEN is_mobile THEN 1.0 ELSE 0 END)
       / COUNT(*) * 100, 2)                                  AS mobile_pct,
     ROUND(AVG(session_duration_minutes), 2)                 AS avg_session_minutes,
     SUM(CASE WHEN is_financial_event  THEN 1 ELSE 0 END)    AS financial_events,
     SUM(CASE WHEN is_long_session     THEN 1 ELSE 0 END)    AS long_sessions
   FROM customer_360.silver.silver_digital_activity
   GROUP BY customer_id
 ),

 payment_metrics AS (
   SELECT
     customer_id,
     COUNT(*)                                                 AS total_payments,
     SUM(CASE WHEN is_missed   THEN 1 ELSE 0 END)            AS missed_payments,
     SUM(CASE WHEN is_late     THEN 1 ELSE 0 END)            AS late_payments,
     SUM(CASE WHEN is_partial  THEN 1 ELSE 0 END)            AS partial_payments,
     SUM(CASE WHEN status = 'on_time' THEN 1 ELSE 0 END)     AS on_time_payments,
     SUM(CASE WHEN is_defaulted THEN 1 ELSE 0 END)           AS defaulted_payments,
     ROUND(
       SUM(CASE WHEN is_missed THEN 1.0 ELSE 0 END)
       / COUNT(*) * 100, 2)                                  AS missed_payment_pct,
     ROUND(AVG(payment_completion_pct), 2)                   AS avg_payment_completion_pct,
     ROUND(SUM(payment_gap), 2)                              AS total_payment_gap
   FROM customer_360.silver.silver_loan_payments
   GROUP BY customer_id
 ),

 notification_metrics AS (
   SELECT
     customer_id,
     COUNT(*)                                                 AS total_notifications,
     SUM(CASE WHEN opened  THEN 1 ELSE 0 END)                AS opened_notifications,
     SUM(CASE WHEN clicked THEN 1 ELSE 0 END)                AS clicked_notifications,
     ROUND(
       SUM(CASE WHEN opened  THEN 1.0 ELSE 0 END)
       / COUNT(*) * 100, 2)                                  AS open_rate,
     ROUND(
       SUM(CASE WHEN clicked THEN 1.0 ELSE 0 END)
       / COUNT(*) * 100, 2)                                  AS click_rate,
     ROUND(AVG(engagement_score), 2)                         AS avg_notification_engagement
   FROM customer_360.silver.silver_notifications
   GROUP BY customer_id
 ),

 interaction_metrics AS (
   SELECT
     customer_id,
     COUNT(*)                                                AS total_interactions,
     SUM(CASE WHEN is_branch_visit THEN 1 ELSE 0 END)        AS branch_visits,
     SUM(CASE WHEN is_complaint    THEN 1 ELSE 0 END)        AS complaints,
     SUM(CASE WHEN is_resolved     THEN 1 ELSE 0 END)        AS resolved_interactions,
     ROUND(
       SUM(CASE WHEN is_resolved THEN 1.0 ELSE 0 END)
       / NULLIF(COUNT(*), 0) * 100, 2)                       AS resolution_rate
   FROM customer_360.silver.silver_customer_interactions
   GROUP BY customer_id
 ),

 assembled AS (
   SELECT
     b.*,
     COALESCE(am.total_accounts,       0)    AS total_accounts,
     COALESCE(am.total_balance,         0)    AS total_balance,
     COALESCE(am.savings_balance,       0)    AS savings_balance,
     COALESCE(am.checking_balance,      0)    AS checking_balance,
     COALESCE(am.business_balance,      0)    AS business_balance,
     COALESCE(am.avg_account_age_days,  0)    AS avg_account_age_days,
     COALESCE(am.top_balance_tier, 'Standard') AS top_balance_tier,

     COALESCE(lm.total_loans,            0)   AS total_loans,
     COALESCE(lm.total_loan_amount,      0)   AS total_loan_amount,
     COALESCE(lm.avg_interest_rate,      0)   AS avg_interest_rate,
     COALESCE(lm.has_high_interest_loan, 0)   AS has_high_interest_loan,
     COALESCE(lm.total_monthly_interest, 0)   AS total_monthly_interest,

     COALESCE(cm.total_cards,              0) AS total_cards,
     COALESCE(cm.active_cards,             0) AS active_cards,
     COALESCE(cm.has_active_credit_card,   0) AS has_active_credit_card,
     COALESCE(cm.has_active_debit_card,    0) AS has_active_debit_card,

     COALESCE(tm.total_transactions,       0) AS total_transactions,
     COALESCE(tm.successful_transactions,  0) AS successful_transactions,
     COALESCE(tm.total_spending,           0) AS total_spending,
     COALESCE(tm.avg_monthly_spending,     0) AS avg_monthly_spending,
     COALESCE(tm.active_months,            0) AS active_months,
     tm.last_transaction_date,
     COALESCE(tm.days_since_last_transaction, 999) AS days_since_last_transaction,
     COALESCE(tm.digital_transactions,     0) AS digital_transactions,
     COALESCE(tm.card_transactions,        0) AS card_transactions,

     COALESCE(dm.total_sessions,           0) AS total_sessions,
     COALESCE(dm.total_logins,             0) AS total_logins,
     COALESCE(dm.mobile_sessions,          0) AS mobile_sessions,
     COALESCE(dm.desktop_sessions,         0) AS desktop_sessions,
     COALESCE(dm.mobile_pct,               0) AS mobile_pct,
     COALESCE(dm.avg_session_minutes,      0) AS avg_session_minutes,
     COALESCE(dm.financial_events,         0) AS financial_events,
     COALESCE(dm.long_sessions,            0) AS long_sessions,

     COALESCE(pm.total_payments,           0) AS total_payments,
     COALESCE(pm.missed_payments,          0) AS missed_payments,
     COALESCE(pm.late_payments,            0) AS late_payments,
     COALESCE(pm.on_time_payments,         0) AS on_time_payments,
     COALESCE(pm.defaulted_payments,       0) AS defaulted_payments,
     COALESCE(pm.missed_payment_pct,       0) AS missed_payment_pct,
     COALESCE(pm.avg_payment_completion_pct, 100) AS avg_payment_completion_pct,
     COALESCE(pm.total_payment_gap,        0) AS total_payment_gap,

     COALESCE(nm.total_notifications,      0) AS total_notifications,
     COALESCE(nm.opened_notifications,     0) AS opened_notifications,
     COALESCE(nm.clicked_notifications,    0) AS clicked_notifications,
     COALESCE(nm.open_rate,                0) AS notification_open_rate,
     COALESCE(nm.click_rate,               0) AS notification_click_rate,
     COALESCE(nm.avg_notification_engagement, 0) AS avg_notification_engagement,

     COALESCE(im.total_interactions,       0) AS total_interactions,
     COALESCE(im.branch_visits,            0) AS branch_visits,
     COALESCE(im.complaints,               0) AS complaints,
     COALESCE(im.resolution_rate,          0) AS resolution_rate,

     COALESCE(am.total_accounts, 0)
       + COALESCE(lm.total_loans, 0)
       + COALESCE(cm.active_cards, 0)                        AS total_products

   FROM base b
   LEFT JOIN account_metrics       am ON b.customer_id = am.customer_id
   LEFT JOIN loan_metrics          lm ON b.customer_id = lm.customer_id
   LEFT JOIN card_metrics          cm ON b.customer_id = cm.customer_id
   LEFT JOIN txn_metrics           tm ON b.customer_id = tm.customer_id
   LEFT JOIN digital_metrics       dm ON b.customer_id = dm.customer_id
   LEFT JOIN payment_metrics       pm ON b.customer_id = pm.customer_id
   LEFT JOIN notification_metrics  nm ON b.customer_id = nm.customer_id
   LEFT JOIN interaction_metrics   im ON b.customer_id = im.customer_id
 ),

 scored AS (
   SELECT
     *,

     ROUND(
       LEAST(total_logins       / 24.0,  1.0) * 30 +
       LEAST(total_transactions / 200.0, 1.0) * 30 +
       LEAST(total_products     / 5.0,   1.0) * 20 +
       LEAST(mobile_pct         / 100.0, 1.0) * 20,
     0)                                                       AS engagement_score,

     ROUND(
       LEAST(missed_payment_pct           / 100.0, 1.0) * 40 +
       LEAST(days_since_last_transaction  / 365.0, 1.0) * 30 +
       LEAST(complaints                   / 5.0,   1.0) * 20 +
       (1.0 - LEAST(avg_notification_engagement / 2.0, 1.0)) * 10,
     0)                                                       AS churn_risk_score
   FROM assembled
 )

 SELECT
   *,

   CASE
     WHEN churn_risk_score < 30 THEN 'Low'
     WHEN churn_risk_score < 60 THEN 'Medium'
     ELSE                            'High'
   END                                                        AS churn_risk_tier,

   CASE
     WHEN total_balance >= 100000 OR credit_rating = 5 THEN 'Platinum'
     WHEN total_balance >= 50000  OR credit_rating = 4 THEN 'Gold'
     WHEN total_balance >= 10000  OR credit_rating = 3 THEN 'Silver'
     ELSE                                                    'Standard'
   END                                                        AS customer_value_tier,

   CASE WHEN total_balance >= 50000
           OR credit_rating >= 4     THEN TRUE ELSE FALSE END AS is_high_value,
   CASE WHEN mobile_pct > 50         THEN TRUE ELSE FALSE END AS is_digital_first,
   CASE WHEN churn_risk_score < 30   THEN TRUE ELSE FALSE END AS is_low_churn_risk,


   current_timestamp()                                        AS gold_processed_time

 FROM scored;
 

In [0]:
%sql
select * from customer_360.gold.customer_360

In [0]:
%sql

 SELECT
   COUNT(*)                           AS total_customers,  
   COUNT(DISTINCT customer_id)        AS unique_customers,
   ROUND(AVG(total_balance), 2)       AS avg_total_balance,
   ROUND(AVG(engagement_score), 1)    AS avg_engagement_score,
   ROUND(AVG(churn_risk_score), 1)    AS avg_churn_risk_score,
   COUNT(CASE WHEN is_high_value      THEN 1 END) AS high_value_customers,
   COUNT(CASE WHEN is_digital_first   THEN 1 END) AS digital_first_customers,
   COUNT(CASE WHEN is_low_churn_risk  THEN 1 END) AS low_churn_customers
 FROM customer_360.gold.customer_360;
 
 


Monthly Spending Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.gold.monthly_spending AS
SELECT
  customer_id,
  year_month,
  transaction_year,
  transaction_month,
  COUNT(*)                                               AS total_transactions,
  SUM(CASE WHEN is_successful THEN 1 ELSE 0 END)         AS successful_transactions,
  ROUND(SUM(spending_amount), 2)                         AS total_spending,
  ROUND(
    AVG(CASE WHEN spending_amount > 0
             THEN spending_amount END), 2)               AS avg_transaction_amount,
  COUNT(DISTINCT merchant_category)                      AS unique_categories,
  COUNT(DISTINCT channel)                                AS unique_channels,
  SUM(CASE WHEN is_digital_channel THEN 1 ELSE 0 END)    AS digital_transactions
FROM customer_360.silver.silver_transactions
WHERE is_debit = TRUE
GROUP BY customer_id, year_month, transaction_year, transaction_month
ORDER BY customer_id, year_month;

In [0]:
%sql
select * from customer_360.gold.monthly_spending

Product Holdings Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.gold.product_holdings AS
SELECT
  customer_id,
  'Savings Account'  AS product_type,
  'Account'          AS product_category,
  ROUND(SUM(balance_usd), 2) AS amount,
  COUNT(*)           AS product_count,
  'Active'           AS product_status,
  1                  AS sort_order
FROM customer_360.silver.silver_accounts
WHERE account_type = 'Savings'
GROUP BY customer_id
UNION ALL
SELECT
  customer_id,
  'Checking Account' AS product_type,
  'Account'          AS product_category,
  ROUND(SUM(balance_usd), 2) AS amount,
  COUNT(*)           AS product_count,
  'Active'           AS product_status,
  2                  AS sort_order
FROM customer_360.silver.silver_accounts
WHERE account_type = 'Checking'
GROUP BY customer_id
UNION ALL

SELECT
  customer_id,
  'Business Account' AS product_type,
  'Account'          AS product_category,
  ROUND(SUM(balance_usd), 2) AS amount,
  COUNT(*)           AS product_count,
  'Active'           AS product_status,
  3                  AS sort_order
FROM customer_360.silver.silver_accounts
WHERE account_type = 'Business'
GROUP BY customer_id
UNION ALL

SELECT
  customer_id,
  'Loan'             AS product_type,
  'Loan'             AS product_category,
  ROUND(SUM(loan_amount), 2) AS amount,
  COUNT(*)           AS product_count,
  'Active'           AS product_status,
  4                  AS sort_order
FROM customer_360.silver.silver_loans
GROUP BY customer_id
UNION ALL

SELECT
  sa.customer_id,
  'Credit Card'      AS product_type,
  'Card'             AS product_category,
  NULL               AS amount,
  COUNT(*)           AS product_count,
  MAX(sc.card_status) AS product_status,
  5                  AS sort_order
FROM customer_360.silver.silver_cards sc
JOIN customer_360.silver.silver_accounts sa ON sc.account_id = sa.account_id
WHERE sc.card_type = 'Credit'
GROUP BY sa.customer_id
UNION ALL

SELECT
  sa.customer_id,
  'Debit Card'       AS product_type,
  'Card'             AS product_category,
  NULL               AS amount,
  COUNT(*)           AS product_count,
  MAX(sc.card_status) AS product_status,
  6                  AS sort_order
FROM customer_360.silver.silver_cards sc
JOIN customer_360.silver.silver_accounts sa ON sc.account_id = sa.account_id
WHERE sc.card_type = 'Debit'
GROUP BY sa.customer_id;

In [0]:
%sql
select * from customer_360.gold.product_holdings

Channel Usage Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.gold.channel_usage AS
SELECT
  customer_id,
  channel,
  device_category,
  COUNT(*)                                               AS sessions,
  ROUND(SUM(session_duration_minutes), 2)               AS total_minutes,
  ROUND(AVG(session_duration_minutes), 2)               AS avg_session_minutes,
  SUM(CASE WHEN is_login           THEN 1 ELSE 0 END)   AS logins,
  SUM(CASE WHEN is_financial_event THEN 1 ELSE 0 END)   AS financial_events,

  ROUND(
    COUNT(*) * 100.0
    / SUM(COUNT(*)) OVER (PARTITION BY customer_id),
  2)                                                     AS session_pct
FROM customer_360.silver.silver_digital_activity
GROUP BY customer_id, channel, device_category;

In [0]:
%sql
select * from customer_360.gold.channel_usage

Churn Risk Monthly Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.gold.churn_risk_monthly AS
SELECT
  customer_id,
  DATE_FORMAT(due_date, 'yyyy-MM')                       AS year_month,
  COUNT(*)                                               AS total_payments,
  SUM(CASE WHEN is_missed   THEN 1 ELSE 0 END)           AS missed_payments,
  SUM(CASE WHEN is_late     THEN 1 ELSE 0 END)           AS late_payments,
  SUM(CASE WHEN is_defaulted THEN 1 ELSE 0 END)          AS defaults,
  ROUND(
    SUM(CASE WHEN is_missed THEN 1.0 ELSE 0 END)
    / COUNT(*) * 100, 2)                                 AS monthly_miss_rate,
  ROUND(
    LEAST(SUM(CASE WHEN is_missed THEN 1.0 ELSE 0 END)
          / COUNT(*), 1.0) * 60 +
    LEAST(SUM(CASE WHEN is_late   THEN 1.0 ELSE 0 END)
          / COUNT(*), 1.0) * 40,
  0)                                                     AS monthly_churn_risk_score
FROM customer_360.silver.silver_loan_payments
GROUP BY customer_id, DATE_FORMAT(due_date, 'yyyy-MM')
ORDER BY customer_id, year_month;

In [0]:
%sql
select * from customer_360.gold.churn_risk_monthly

Activity Feed Gold Layer

In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.gold.activity_feed AS

SELECT
  customer_id,
  transaction_datetime                                   AS event_datetime,
  transaction_type                                       AS event_type,
  'Transaction'                                          AS event_category,
  CONCAT(
    INITCAP(REPLACE(transaction_type, '_', ' ')),
    CASE WHEN merchant_category IS NOT NULL
         THEN CONCAT(' — ', INITCAP(merchant_category))
         ELSE '' END
  )                                                      AS event_description,
  amount,
  channel,
  CASE WHEN is_debit THEN 'Debit' ELSE 'Credit' END      AS direction,
  status                                                 AS event_status,
  transaction_date                                       AS event_date
FROM customer_360.silver.silver_transactions
WHERE is_successful = TRUE
UNION ALL

SELECT
  customer_id,
  interaction_datetime                                   AS event_datetime,
  interaction_type                                       AS event_type,
  'Interaction'                                          AS event_category,
  CONCAT(
    INITCAP(REPLACE(interaction_type, '_', ' ')),
    ' — ',
    COALESCE(subject, channel)
  )                                                      AS event_description,
  NULL                                                   AS amount,
  channel,
  NULL                                                   AS direction,
  status                                                 AS event_status,
  interaction_date                                       AS event_date
FROM customer_360.silver.silver_customer_interactions;

In [0]:
%sql
select * from customer_360.gold.activity_feed

In [0]:
%sql
SELECT
  customer_id,
  full_name,
  city,
  total_balance,
  total_products,
  engagement_score,
  churn_risk_tier,
  total_transactions,
  total_sessions
FROM customer_360.gold.customer_360
WHERE total_transactions  > 50
  AND total_sessions      > 20
  AND total_loans         > 0
  AND active_cards        > 0
  AND is_low_churn_risk   = TRUE
ORDER BY engagement_score DESC
LIMIT 20;

HIGH VALUE customer (best for impressing executives)

In [0]:
%sql
SELECT customer_id, full_name, total_balance, engagement_score, churn_risk_tier
FROM customer_360.gold.customer_360
WHERE total_transactions > 0
  AND total_sessions     > 0
  AND total_payments     > 0
  AND total_interactions > 0
  AND customer_value_tier = 'Platinum'
  AND is_low_churn_risk  = TRUE
ORDER BY total_balance DESC
LIMIT 5;

HIGH RISK customer (good for showing churn risk panel)

In [0]:
%sql
SELECT customer_id, full_name, total_balance, churn_risk_score, missed_payment_pct
FROM customer_360.gold.customer_360
WHERE total_transactions > 0
  AND total_sessions     > 0
  AND total_payments     > 0
  AND total_interactions > 0
  AND churn_risk_tier    = 'High'
ORDER BY churn_risk_score DESC
LIMIT 5;

DIGITAL FIRST customer (good for showing channel usage panel)

In [0]:
%sql
SELECT customer_id, full_name, mobile_pct, total_sessions, engagement_score
FROM customer_360.gold.customer_360
WHERE total_transactions > 0
  AND total_sessions     > 0
  AND total_payments     > 0
  AND total_interactions > 0
  AND is_digital_first   = TRUE
ORDER BY mobile_pct DESC, total_sessions DESC
LIMIT 5;